In [1]:
!pip install faiss-cpu sentence-transformers rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 100.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 5.8 MB/s eta 0:00:000:00:0100:01
  Attempting un

In [2]:
import os
import pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList
from peft import PeftModel
import time
import gc

2025-11-29 13:47:01.588208: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764424021.761026      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764424021.810553      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
class VietnameseRAGSystem:
    def __init__(self, embedding_model_name="intfloat/multilingual-e5-large-instruct", cross_encoder_name="namdp-ptit/ViRanker", cache_dir="/kaggle/working", device=None):
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.device = device

        print("Loading embedding model with optimizations...")
        model_kwargs = {'torch_dtype': torch.float16} if device == 'cuda' else {}
        self.embedding_model = SentenceTransformer(
            embedding_model_name, 
            device=device,
            model_kwargs=model_kwargs,
            cache_folder=cache_dir
        )
        self.embedding_model.eval()
        if device == 'cuda':
            self.embedding_model.half()
        
        torch.cuda.empty_cache()

        print("Loading cross-encoder model with optimizations...")
        self.cross_encoder = CrossEncoder(cross_encoder_name, device=device)
        if device == 'cuda':
            self.cross_encoder.model.half()

        self.index = None
        self.chunks = []
        self.metadata = []
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
       
        self.retrieval_task = 'Given a query about Vietnamese history, retrieve relevant historical passages that answer the query'
        self.embedding_cache = {}
   
    def normalize_text(self, text):
        return text.lower()
   
    def get_detailed_instruct(self, query):
        return f'Instruct: {self.retrieval_task}\nQuery: {query}'  # Sửa lỗi \\n -> \n
   
    def load_index(self, faiss_file, metadata_file):
        # Sử dụng cache cho index
        cache_path = os.path.join(self.cache_dir, "index_cache.pkl")
        if os.path.exists(cache_path):
            print("Loading index from cache...")
            with open(cache_path, 'rb') as f:
                cached_data = pickle.load(f)
                self.index = cached_data['index']
                self.chunks = cached_data['chunks']
                self.metadata = cached_data['metadata']
                print(f"Loaded cached index with {len(self.chunks)} chunks")
                return
        
        # Tải index gốc
        self.index = faiss.read_index(faiss_file)
        with open(metadata_file, 'rb') as f:
            index_data = pickle.load(f)
       
        self.chunks = index_data['chunks']
        self.metadata = index_data['metadata']
       
        # Lưu cache
        with open(cache_path, 'wb') as f:
            pickle.dump({
                'index': self.index,
                'chunks': self.chunks,
                'metadata': self.metadata
            }, f)
       
        print(f"Loaded index with {len(self.chunks)} chunks")
   
    def hybrid_search(self, query, top_k=5, filter_trieu_dai=None, filter_chu_de=None):
        if self.index is None:
            raise ValueError("Index not initialized!")
       
        # Sử dụng cache cho kết quả tìm kiếm
        cache_key = f"{query}_{filter_trieu_dai}_{filter_chu_de}_{top_k}"
        if cache_key in self.embedding_cache:
            return self.embedding_cache[cache_key]
       
        initial_results = self._semantic_search(
            query,
            top_k=min(50, len(self.chunks)),
            filter_trieu_dai=filter_trieu_dai,
            filter_chu_de=filter_chu_de
        )
       
        if not initial_results:
            return []
       
        reranked_results = self._rerank_with_cross_encoder(query, initial_results, top_k=top_k)
        self.embedding_cache[cache_key] = reranked_results  # Lưu cache
        return reranked_results
    
    def _semantic_search(self, query, top_k=10, filter_trieu_dai=None, filter_chu_de=None):
        instructed_query = self.get_detailed_instruct(query)
        cache_key = f"embed_{instructed_query}"
        
        # Sử dụng cache cho embedding
        if cache_key in self.embedding_cache:
            query_embedding = self.embedding_cache[cache_key]
        else:
            with torch.no_grad():
                query_embedding = self.embedding_model.encode(
                    [instructed_query], 
                    normalize_embeddings=True,
                    convert_to_tensor=True
                ).cpu().numpy()
            self.embedding_cache[cache_key] = query_embedding
        
        candidate_size = min(top_k * 3, len(self.chunks))
        semantic_scores, semantic_indices = self.index.search(query_embedding, candidate_size)
        
        results = []
        for score, idx in zip(semantic_scores[0], semantic_indices[0]):
            if idx >= len(self.chunks) or score < 0.1:
                continue
                
            metadata = self.metadata[idx]
            
            # Áp dụng bộ lọc
            if filter_trieu_dai and metadata.get('trieu_dai') != filter_trieu_dai:
                continue
            if filter_chu_de and metadata.get('chu_de') != filter_chu_de:
                continue
            
            results.append({
                'text': self.chunks[idx]['text'],
                'metadata': metadata,
                'score': float(score),
                'original_index': int(idx)
            })
            
            if len(results) >= top_k:
                break
        
        return results
   
    def _rerank_with_cross_encoder(self, query, initial_results, top_k=5):
        if not initial_results:
            return []
       
        max_text_length = 384  # Giảm độ dài để tăng tốc
        documents = [result['text'][:max_text_length] for result in initial_results]
        pairs = [[query, doc] for doc in documents]
       
        try:
            # Tăng batch size và tắt progress bar
            ce_scores = self.cross_encoder.predict(
                pairs, 
                batch_size=64,  # Tăng từ 32 lên 64
                show_progress_bar=False,
                convert_to_tensor=True
            ).cpu().numpy()
        except Exception as e:
            print(f"Cross-encoder error: {e}. Use initial results.")
            return initial_results[:top_k]
       
        final_results = []
        for i, (ce_score, original_result) in enumerate(zip(ce_scores, initial_results)):
            semantic_score = original_result['score']
            combined_score = 0.65 * ce_score + 0.35 * semantic_score  # Điều chỉnh trọng số
           
            final_results.append({
                'text': original_result['text'],
                'metadata': original_result['metadata'],
                'score': float(combined_score),
                'ce_score': float(ce_score),
                'original_score': semantic_score
            })
       
        final_results.sort(key=lambda x: x['score'], reverse=True)
        return final_results[:top_k]
   
    def search(self, query, top_k=5, filter_trieu_dai=None, filter_chu_de=None):
        return self.hybrid_search(query, top_k, filter_trieu_dai, filter_chu_de)
   
    def get_available_filters(self):
        trieu_dais = set(m.get('trieu_dai') for m in self.metadata if m.get('trieu_dai'))
        chu_des = set(m.get('chu_de') for m in self.metadata if m.get('chu_de'))
       
        return {
            'trieu_dai': sorted(list(trieu_dais)),
            'chu_de': sorted(list(chu_des))
        }

In [4]:
rag_system = VietnameseRAGSystem()
faiss_cache_file = "/kaggle/input/vectordb/rag_index.faiss"
metadata_cache_file = "/kaggle/input/vectordb/rag_metadata.pkl"

rag_system.load_index(faiss_cache_file, metadata_cache_file)

filters = rag_system.get_available_filters()
print(f"RAG system ready with {len(rag_system.chunks)} documents")
print(f"Available dynasties: {len(filters['trieu_dai'])}")
print(f"Available topics: {len(filters['chu_de'])}")

Loading embedding model with optimizations...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loaded index with 19509 chunks
RAG system ready with 19509 documents
Available dynasties: 1118
Available topics: 18882


In [5]:
model_id = "Qwen/Qwen3-4B"
adapter_dir = "/kaggle/input/qwen-finetuned/transformers/default/3/qwen_finetuned"

os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# Sử dụng tokenizer nhanh
tokenizer = AutoTokenizer.from_pretrained(
    adapter_dir,
    use_fast=True,  # Kích hoạt tokenizer nhanh
    padding_side="left"
)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    use_cache=True,
    low_cpu_mem_usage=True
)

model = PeftModel.from_pretrained(base_model, adapter_dir)
model = model.merge_and_unload()
model.eval()

# Kích hoạt JIT compilation cho GPU
if torch.cuda.is_available():
    try:
        model = torch.compile(model, mode="reduce-overhead", fullgraph=True)
        print("Model compiled with torch.compile for optimization")
    except Exception as e:
        print(f"Compilation failed: {e}. Using original model.")

if hasattr(model, 'config'):
    model.config.use_cache = True

print("Model and tokenizer loaded successfully")
print(f"Model device: {next(model.parameters()).device}")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model compiled with torch.compile for optimization
Model and tokenizer loaded successfully
Model device: cuda:0


In [17]:
import re
import threading
import time
from transformers import TextIteratorStreamer
import gc
import torch

# Cấu hình
MAX_CLASSIFICATION_TOKENS = 8
RETRIEVAL_THRESHOLD = 0.60

def clear_memory():
    """Dọn dẹp bộ nhớ tổng hợp"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

class QuestionClassifier:
    def __init__(self, tokenizer, model, device):
        self.tokenizer = tokenizer
        self.model = model
        self.device = device
        self.classification_cache = {}
    
    def classify_question(self, question: str) -> str:
        # Kiểm tra cache trước
        if question in self.classification_cache:
            return self.classification_cache[question]
            
        classification_prompt = f"""PHÂN LOẠI CÂU HỎI: Chọn MỘT trong 4 loại dưới đây:
            
1. HISTORY_DIRECT: Câu hỏi CỤ THỂ về sự kiện, nhân vật, thời gian lịch sử Việt Nam (có chi tiết rõ ràng)
2. OUT_OF_DOMAIN: Câu hỏi HOÀN TOÀN KHÔNG LIÊN QUAN đến lịch sử/văn hóa Việt Nam (thời tiết, thể thao, ẩm thực, công nghệ hiện đại)
3. INSUFFICIENT_INFO: Câu hỏi về lịch sử Việt Nam nhưng THÔNG TIN KHÔNG TỒN TẠI (ví dụ: công nghệ hiện đại trong thời phong kiến, phát minh không có thật)
4. VAGUE: Câu hỏi QUÁ RỘNG, MƠ HỒ, THIẾU CHI TIẾT (cần làm rõ)

QUAN TRỌNG: 
- INSUFFICIENT_INFO: vẫn là câu hỏi về lịch sử, nhưng thông tin không có thật
- OUT_OF_DOMAIN: không phải câu hỏi về lịch sử

VÍ DỤ:
- "Vua Quang Trung đánh quân Thanh năm nào?" -> HISTORY_DIRECT  
- "Thời tiết Hà Nội thế nào?" -> OUT_OF_DOMAIN
- "Nhà Trần có dùng điện thoại không?" -> INSUFFICIENT_INFO (về lịch sử nhưng không có thật)
- "Triều Nguyễn có Internet không?" -> INSUFFICIENT_INFO  
- "Kể về lịch sử Việt Nam" -> VAGUE
- "Cristiano Ronaldo là ai?" -> OUT_OF_DOMAIN

CÂU HỎI CẦN PHÂN LOẠI: {question}

CHỈ TRẢ LỜI MỘT TỪ: HISTORY_DIRECT, OUT_OF_DOMAIN, INSUFFICIENT_INFO, hoặc VAGUE
Kết quả:"""

        inputs = self.tokenizer(classification_prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        try:
            with torch.inference_mode():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=MAX_CLASSIFICATION_TOKENS,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )

            resp = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            if resp.startswith(classification_prompt):
                classification = resp[len(classification_prompt):].strip()
            else:
                if "Kết quả:" in resp:
                    classification = resp.split("Kết quả:")[-1].strip()
                else:
                    classification = resp.strip()

            first_token = classification.split()[0] if classification else ""
            candidate = first_token.upper().strip().strip('":,.')
            
            if "HISTORY" in candidate:
                result = "HISTORY_DIRECT"
            elif "OUT" in candidate or "DOMAIN" in candidate:
                result = "OUT_OF_DOMAIN"
            elif "INSUFFICIENT" in candidate or "INSUFFICIENT_INFO" in candidate:
                result = "INSUFFICIENT_INFO"
            elif "VAGUE" in candidate:
                result = "VAGUE"
            else:
                c_up = classification.upper()
                if "HISTORY_DIRECT" in c_up:
                    result = "HISTORY_DIRECT"
                elif "OUT_OF_DOMAIN" in c_up or "OUT-OF-DOMAIN" in c_up:
                    result = "OUT_OF_DOMAIN"
                elif "INSUFFICIENT" in c_up:
                    result = "INSUFFICIENT_INFO"
                elif "VAGUE" in c_up:
                    result = "VAGUE"
                else:
                    print(f"Không phân loại rõ: '{classification}' -> fallback HISTORY_DIRECT")
                    result = "HISTORY_DIRECT"

            # Lưu vào cache
            self.classification_cache[question] = result
            return result
            
        finally:
            # Dọn dẹp memory sau mỗi lần classify
            del inputs, outputs
            clear_memory()

class QASystemWithClassification:
    def __init__(self, tokenizer, model, device, rag_system):
        self.tokenizer = tokenizer
        self.model = model
        self.device = device
        self.rag_system = rag_system
        self.classifier = QuestionClassifier(tokenizer, model, device)
        
        # Tối ưu model cho inference
        self.model.eval()
        if hasattr(self.model, 'config'):
            self.model.config.use_cache = True
    
    def generate_answer(self, question: str) -> str:
        try:
            question_type = self.classifier.classify_question(question)
            print(f"Phân loại: {question_type}")
            
            if question_type == "OUT_OF_DOMAIN":
                return self._handle_out_of_domain(question)
            elif question_type == "VAGUE":
                return self._handle_vague_question(question)
            elif question_type == "INSUFFICIENT_INFO":
                return self._handle_insufficient_info(question)
            else:  # HISTORY_DIRECT
                return self._handle_history_question(question)
        finally:
            # Dọn dẹp memory sau mỗi câu hỏi
            clear_memory()
    
    def _handle_out_of_domain(self, question: str) -> str:
        response_prompt = f"""Bạn là trợ lý chuyên gia về lịch sử Việt Nam. Hãy trả lời câu hỏi sau:

Câu hỏi: {question}

Đây là câu hỏi KHÔNG thuộc phạm vi lịch sử Việt Nam. Bạn cần:

1. Thông báo lịch sự rằng đây không phải câu hỏi về lịch sử Việt Nam
2. Giải thích ngắn gọn rằng bạn chỉ có thể hỗ trợ các câu hỏi về lịch sử Việt Nam
3. Mời người dùng hỏi về các chủ đề lịch sử Việt Nam khác

Hãy trả lời một cách chuyên nghiệp và lịch sự. Kết thúc bằng (end)

Trả lời:"""
        
        return self._generate_response_with_streaming(response_prompt)
    
    def _handle_vague_question(self, question: str) -> str:
        response_prompt = f"""Bạn là trợ lý chuyên gia về lịch sử Việt Nam. Hãy trả lời câu hỏi sau:

Câu hỏi: {question}

Đây là câu hỏi QUÁ RỘNG và cần được LÀM RÕ. Bạn cần:

1. Nhận xét rằng câu hỏi này cần được cụ thể hóa
2. Đề nghị người dùng cung cấp thêm chi tiết về: thời gian, địa điểm, nhân vật hoặc sự kiện cụ thể
3. Đưa ra 1-2 ví dụ về cách hỏi cụ thể hơn
4. Giữ thái độ tích cực và hỗ trợ

Hãy trả lời một cách hữu ích. Kết thúc bằng (end)

Trả lời:"""
        
        return self._generate_response_with_streaming(response_prompt)
    
    def _handle_insufficient_info(self, question: str) -> str:
        context = self._get_context_for_question(question)
        if context:
            response_prompt = f"""Bạn là chuyên gia lịch sử Việt Nam. Dưới đây là dữ liệu tham khảo từ hệ thống:

{context}

Câu hỏi: {question}

Đây là câu hỏi về thông tin CÓ THỂ KHÔNG TỒN TẠI trong lịch sử. Bạn cần:

1. Phân tích tính hợp lý của câu hỏi trong bối cảnh lịch sử
2. Trả lời dựa trên dữ liệu có sẵn nếu có thông tin phù hợp
3. Nếu không có thông tin, giải thích rõ ràng tại sao câu hỏi này có thể không có câu trả lời
4. Cung cấp thông tin lịch sử liên quan nếu có thể
5. TUYỆT ĐỐI KHÔNG bịa đặt thông tin

Hãy trả lời một cách trung thực và chính xác. Kết thúc bằng (end)

Trả lời:"""
        else:
            response_prompt = f"""Bạn là chuyên gia lịch sử Việt Nam.

Câu hỏi: {question}

Đây là câu hỏi về thông tin CÓ THỂ KHÔNG TỒN TẠI trong lịch sử. Bạn cần:

1. Phân tích tính hợp lý của câu hỏi trong bối cảnh lịch sử
2. Giải thích rõ ràng tại sao thông tin này có thể không tồn tại
3. Cung cấp thông tin lịch sử liên quan nếu có thể
4. Không bịa đặt thông tin

Hãy trả lời một cách trung thực. Kết thúc bằng (end)

Trả lời:"""
        
        return self._generate_response_with_streaming(response_prompt)
    
    def _handle_history_question(self, question: str) -> str:
        context = self._get_context_for_question(question)
        if context:
            response_prompt = f"""Bạn là chuyên gia lịch sử Việt Nam. Dưới đây là DỮ LIỆU TRUY XUẤT ĐƯỢC từ hệ thống (chỉ sử dụng thông tin từ đây để trả lời):

{context}

Câu hỏi: {question}

YÊU CẦU NGHIÊM NGẶT:
✅ Trả lời ĐẦY ĐỦ, CHÍNH XÁC dựa hoàn toàn trên tài liệu trên
✅ Tập trung vào thông tin trực tiếp liên quan đến câu hỏi
✅ Sắp xếp thông tin một cách logic và dễ hiểu
✅ Có thể sử dụng định dạng để làm nổi bật thông tin quan trọng

❌ Nếu tài liệu KHÔNG ĐỦ thông tin, hãy THỪA NHẬN RÕ RÀNG rằng "Tôi không có đủ thông tin để trả lời câu hỏi này"
❌ TUYỆT ĐỐI KHÔNG thêm bất kỳ thông tin nào không có trong dữ liệu
❌ KHÔNG sử dụng tiếng Anh trong câu trả lời

Hãy trả lời bằng tiếng Việt có dấu đầy đủ. Kết thúc bằng (end)

Bắt đầu trả lời:"""
        else:
            response_prompt = f"""Bạn là chuyên gia lịch sử Việt Nam.

Câu hỏi: {question}

THÔNG BÁO: Không tìm thấy dữ liệu phù hợp trong hệ thống để trả lời câu hỏi này.

Bạn cần:
1. Thông báo lịch sự rằng không có đủ thông tin để trả lời
2. Đề xuất người dùng hỏi câu hỏi khác về lịch sử Việt Nam
3. Giữ thái độ hỗ trợ và chuyên nghiệp

Hãy trả lời một cách lịch sự. Kết thúc bằng (end)

Trả lời:"""
        
        return self._generate_response_with_streaming(response_prompt)
    
    def _get_context_for_question(self, question: str, top_k: int = 4) -> str:
        # Giảm số lượng kết quả retrieval để tăng tốc
        results = self.rag_system.search(question, top_k=min(20, top_k*2))
        
        if not results:
            print("Không có kết quả trả về từ retrieval.")
            return None

        top_score = results[0].get('score', 0.0)
        print(f"Điểm số top result: {top_score:.3f}")
        
        if top_score < RETRIEVAL_THRESHOLD:
            print(f"Điểm số thấp hơn ngưỡng {RETRIEVAL_THRESHOLD}, không sử dụng context.")
            return None

        # Chỉ lấy 2 kết quả tốt nhất để giảm context length
        top_results = results[:min(4, len(results))]
        context_parts = []
        for i, result in enumerate(top_results, 1):
            text = result['text'].strip()
            score = result.get('score', 0.0)
            dynasty = result.get('metadata', {}).get('trieu_dai', 'Không rõ')
            print(f"Kết quả {i}: Điểm={score:.3f}, Triều đại={dynasty}")
            context_parts.append(text)

        return "\n\n".join(context_parts)
    
    def _generate_response_with_streaming(self, prompt: str) -> str:
        inputs = None
        try:
            # Giảm max_length của prompt để tăng tốc
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            # Tạo streamer với timeout ngắn hơn
            streamer = TextIteratorStreamer(self.tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=15)
            
            full_output = ""
            stop_generation = False
            
            def generate_with_streamer(inputs_dict):
                try:
                    with torch.inference_mode():
                        # Tối ưu generation parameters để tăng tốc
                        self.model.generate(
                            **inputs_dict,
                            max_new_tokens=512,
                            do_sample=True,
                            temperature=0.3,
                            streamer=streamer,
                            pad_token_id=self.tokenizer.eos_token_id
                        )
                except Exception as e:
                    print(f"Generation error: {e}")
            
            # Chạy generation trong thread riêng
            generation_thread = threading.Thread(target=generate_with_streamer, args=(inputs,))
            generation_thread.start()
            
            # Streaming output
            print("Model đang trả lời: ", end="", flush=True)
            
            # Thu thập output từ streamer
            try:
                for new_text in streamer:
                    if stop_generation:
                        break
                        
                    # In từng phần text khi nhận được
                    print(new_text, end="", flush=True)
                    full_output += new_text
                    
                    # Kiểm tra nếu có bất kỳ biến thể nào của end trong output hiện tại
                    if re.search(r'\(?end\)?', full_output, re.IGNORECASE):
                        stop_generation = True
                        break
                        
            except Exception as e:
                print(f"Streaming error: {e}")
            
            # Đảm bảo thread kết thúc với timeout ngắn hơn
            generation_thread.join(timeout=3)
            
            print()  # Xuống dòng sau khi hoàn thành streaming
            
            # Clean model output - giữ nguyên format như cũ
            clean_output = _clean_model_output(full_output)
            
            return clean_output
            
        finally:
            # Luôn dọn dẹp memory
            if inputs is not None:
                del inputs
            clear_memory()

def _clean_model_output(raw_text: str) -> str:
    """Làm sạch output model với xử lý end linh hoạt"""
    if not raw_text or not raw_text.strip():
        return ""
    
    text = raw_text.strip()
    
    # Tìm và cắt bỏ mọi biến thể của end (không phân biệt hoa thường)
    end_match = re.search(r'\(?end\)?', text, re.IGNORECASE)
    if end_match:
        text = text[:end_match.start()].strip()
    
    # Giữ nguyên tất cả markdown và định dạng
    # Chỉ loại bỏ các code blocks lớn nếu có
    text = re.sub(r'```.*?```', '', text, flags=re.DOTALL)
    
    # Đảm bảo không quá ngắn
    if len(text.strip()) < 10:
        return ""
    
    return text.strip()

In [ ]:
# ========== ALTERNATIVE: STREAMING CHAT INTERFACE ==========
def streaming_chat_interface(qa_system):
    """Giao diện chat với streaming response"""
    print("🎯 STREAMING CHAT - HỆ THỐNG HỎI ĐÁP LỊCH SỬ")
    print("=" * 50)
    print("Gõ câu hỏi và xem kết quả streaming...")
    print("Thoát: 'quit', 'exit', 'stop'")
    print("=" * 50)
    
    while True:
        print("\n" + "─" * 40)
        question = input("🙋 Câu hỏi: ").strip()
        
        if question.lower() in ['quit', 'exit', 'stop', 'q']:
            print("👋 Hẹn gặp lại!")
            break
            
        if not question:
            continue
            
        print("💭 Phân loại...", end=" ")
        start_time = time.time()
        
        try:
            # Phân loại trước
            question_type = qa_system.classifier.classify_question(question)
            print(f"✅ {question_type}")
            
            # Xử lý và stream response
            print("🤖 Trả lời: ", end="")
            
            if question_type == "OUT_OF_DOMAIN":
                answer = qa_system._handle_out_of_domain(question)
            elif question_type == "VAGUE":
                answer = qa_system._handle_vague_question(question)
            elif question_type == "INSUFFICIENT_INFO":
                answer = qa_system._handle_insufficient_info(question)
            else:
                answer = qa_system._handle_history_question(question)
                
            processing_time = time.time() - start_time
            print(f"\n⏱️ Hoàn thành trong {processing_time:.2f}s")
            
        except Exception as e:
            print(f"\n Lỗi: {e}")

# Chạy streaming interface
print("\nKhởi động Streaming Chat Interface...")
streaming_chat_interface(qa_system)


Khởi động Streaming Chat Interface...
🎯 STREAMING CHAT - HỆ THỐNG HỎI ĐÁP LỊCH SỬ
Gõ câu hỏi và xem kết quả streaming...
Thoát: 'quit', 'exit', 'stop'

────────────────────────────────────────
